In [ ]:
import logging
import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
import shutil
from pathlib import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
%load_ext autoreload
%autoreload 2
import air2waterwrapper as a2w
from scipy.stats import spearmanr

#set up logging and correct file path
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)
print(a2w.PERIODS)        # what's ACTUALLY loaded in memory right now?

figure_path = os.path.join(a2w.REPO_ROOT, 'images', 'temperature_estimation') + os.sep

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 11, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})



In [ ]:
#id=20853 Fort Peck (28.9 m, 814 km² 
a2w.make_dirs(20853)

In [ ]:
a2w.write_input_txt(20853)

In [ ]:
a2w.write_pso_txt(20853)

In [ ]:
df = a2w.load_lake_series(20853)

In [ ]:
df

In [ ]:
a2w.write_data_file(20853, df, 'cal')
a2w.write_data_file(20853, df, 'val')

In [ ]:
par = a2w.air2water_param_bounds(147)

In [ ]:
a2w.make_parameters_txt(20853, depth=28.9)

In [ ]:
a2w.run_air2water(20853)

In [ ]:
ids = pd.read_csv(a2w.LWST_CSV)['id_v11'].astype(int).tolist()
results, failures = {}, {}
for dam_id in ids:
    try:
        a2w.make_dirs(dam_id)
        a2w.write_input_txt(dam_id)
        a2w.write_pso_txt(dam_id)
        d = a2w.load_lake_series(dam_id)
        a2w.write_data_file(dam_id, d, 'cal')
        a2w.write_data_file(dam_id, d, 'val')
        a2w.make_parameters_txt(dam_id)          # looks up depth itself
        a2w.run_air2water(dam_id)
        results[dam_id] = a2w.read_output(dam_id)
        logger.info("done %s  RMSE_val=%.3f", dam_id, results[dam_id]['eff_val'])
    except Exception as e:
        failures[dam_id] = str(e)
        logger.warning("FAILED %s: %s", dam_id, e)

print(f"done {len(results)}, failed {len(failures)}")

In [ ]:
ids = pd.read_csv(a2w.LWST_CSV)['id_v11'].astype(int).tolist()
results, failures = {}, {}
for dam_id in ids:
    try:
        results[dam_id] = a2w.read_output(dam_id)
        logger.info("done %s  RMSE_val=%.3f", dam_id, results[dam_id]['eff_val'])
    except Exception as e:
        failures[dam_id] = str(e)
        logger.warning("FAILED %s: %s", dam_id, e)

In [ ]:
results

In [ ]:
pd.read_csv('validation_metrics_temp.csv')

In [ ]:
def nse(o, s):  return 1 - np.sum((s-o)**2) / np.sum((o-o.mean())**2)
def bias(o, s): return np.mean(s-o)

meta = pd.read_csv(a2w.LWST_CSV); meta['id_v11'] = meta['id_v11'].astype(int)

rows = []
for did, r in results.items():
    v = r['val_series'].dropna(subset=['obs_water', 'sim_water'])
    o, s = v['obs_water'].to_numpy(), v['sim_water'].to_numpy()
    m = meta[meta.id_v11 == int(did)].iloc[0]
    rows.append({
        'dam_id': int(did), 'lake': m['Lake_name'],
        'continent': m['Continent'], 'depth_m': m['Depth_avg'],
        'rmse_cal': round(r['eff_cal'], 3),       # calibration RMSE (from exe, 1_PSO line 2)
        'rmse_val': round(r['eff_val'], 3),       # validation  RMSE (from exe, 1_PSO line 3)
        'nse_val':  round(nse(o, s), 3),
        'bias_val': round(bias(o, s), 3),
        'n_val':    len(v),
        'obs_std':  round(o.std(), 2),
        'obs_min':  round(o.min(), 1),
        'obs_max':  round(o.max(), 1),
        **{f'p{i+1}': round(r['params'][i], 6) for i in range(8)},
    })

metrics = pd.DataFrame(rows).sort_values('rmse_val')
metrics.to_csv(a2w.HERE / 'validation_metrics_temp.csv', index=False)
print(f"{len(metrics)} dams | median RMSE_val {metrics.rmse_val.median():.3f} | median NSE {metrics.nse_val.median():.3f}")

In [ ]:
metrics['rmse_cal'].mean()

In [ ]:
metrics['rmse_val'].mean()

In [ ]:
long = []
for did, r in results.items():
    for period, key in [('cal', 'cal_series'), ('val', 'val_series')]:
        d = r[key][['date', 'obs_water', 'sim_water']].copy()
        d['dam_id'] = did
        d['period'] = period
        long.append(d)
long = pd.concat(long, ignore_index=True).dropna(subset=['obs_water', 'sim_water'])

In [ ]:
long

In [ ]:
v = long[long.period == 'val']
plt.figure(figsize=(5, 5))
plt.hexbin(v.obs_water, v.sim_water, gridsize=60, mincnt=1, cmap='inferno')
lims = [v.obs_water.min(), v.obs_water.max()]
plt.plot(lims, lims, 'k--', lw=1)                 # 1:1 line
plt.xlabel('Observed LSWT (°C)'); plt.ylabel('Modelled LSWT (°C)')
plt.colorbar(label='point density'); plt.gca().set_aspect('equal')
plt.tight_layout(); plt.show()

In [ ]:

lims = [long.obs_water.min(), long.obs_water.max()]      # common range for both panels

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
hbs = []
for ax, period, title in zip(axes, ['cal', 'val'], ['Calibration', 'Validation']):
    d = long[long.period == period]
    hb = ax.hexbin(d.obs_water, d.sim_water, gridsize=60, mincnt=1, cmap='inferno',
                   extent=(lims[0], lims[1], lims[0], lims[1]))
    ax.plot(lims, lims, 'k--', lw=1)                     # 1:1 line
    ax.set_aspect('equal'); ax.set_xlabel('Observed LSWT (°C)')
    rmse = np.sqrt(np.mean((d.sim_water - d.obs_water) ** 2))
    ax.set_title(f'{title}   (RMSE {rmse:.2f} °C, n={len(d):,})')
    hbs.append(hb)

vmax = max(hb.get_array().max() for hb in hbs)           # unify the density color scale
for hb in hbs: hb.set_clim(0, vmax)

axes[0].set_ylabel('Modelled LSWT (°C)')
fig.colorbar(hbs[-1], ax=axes, label='point density', shrink=0.8)
plt.savefig(figure_path + 'ParityplotTemp.pdf')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

picks = {20853: 'Fort Peck — N. America, deep, strongly seasonal',
         22428: 'Guárico — S. America, tropical, low variance',
         23425: 'Yardi — Africa, weakest fit'}

fig, axes = plt.subplots(len(picks), 1, figsize=(11, 10), sharex=True)
for ax, (did, label) in zip(axes, picks.items()):
    d = long[(long.dam_id == did) & (long.period == 'val') & long.date.dt.year.between(2008, 2012)]
    ax.plot(d.date, d.obs_water, '.', ms=3, color='tab:grey', label='observed (GloboLakes)')
    ax.plot(d.date, d.sim_water, '-', lw=0.8, color='tab:red', label='air2water')
    m = metrics[metrics.dam_id == did].iloc[0]
    ax.set_title(f"{label}   —   RMSE {m.rmse_val:.2f} °C, NSE {m.nse_val:.2f}", fontsize=10)
    ax.set_ylabel('LSWT (°C)')
axes[0].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig(figure_path + 'hydrographs3fits.pdf')
plt.show()

In [ ]:
# attach coordinates to the metrics table
geo = metrics.merge(meta[['id_v11', 'lat', 'lon']], left_on='dam_id', right_on='id_v11', how='left')

fig = plt.figure(figsize=(11, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-170, 180, -50, 72], ccrs.PlateCarree())
ax.add_feature(cfeature.OCEAN.with_scale('110m'), facecolor='#dbeaf2')
ax.add_feature(cfeature.LAND.with_scale('110m'),  facecolor='#f6f4ee')
ax.add_feature(cfeature.LAKES.with_scale('110m'), facecolor='#dbeaf2', edgecolor='none')
ax.add_feature(cfeature.COASTLINE.with_scale('110m'), linewidth=.5)
ax.add_feature(cfeature.BORDERS.with_scale('110m'), edgecolor='gray', linewidth=.3)

# color = validation RMSE (the result)
sc = ax.scatter(geo.lon, geo.lat, c=geo.rmse_val, cmap='inferno',
                s=30, edgecolor='white', linewidth=.4, zorder=5,
                transform=ccrs.PlateCarree())
plt.colorbar(sc, label='validation RMSE [°C]', shrink=.5)

gl = ax.gridlines(draw_labels=True, linewidth=.3, color='gray', alpha=.4)
gl.top_labels = False; gl.right_labels = False

ax.text(0.99, 0.01, 'CRS: WGS84 (EPSG:4326), PlateCarree',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=7, color='gray')

plt.title('56 validation lakes, coloured by daily LSWT validation RMSE')
plt.tight_layout()
plt.savefig(figure_path +'lakeTempMap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
g = metrics.merge(meta[['id_v11', 'lat']], left_on='dam_id', right_on='id_v11', how='left')
g['abslat'] = g['lat'].abs()

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(g.depth_m, g.rmse_val, c=g.abslat, cmap='inferno',
                s=45, edgecolor='k', linewidth=.3)
ax.set_xscale('log')                                   # depth is right-skewed (~3–150 m)
ax.set_xlabel('Mean depth [m]'); ax.set_ylabel('Validation RMSE [°C]')
plt.colorbar(sc, label='|latitude| [°]')

rho, p = spearmanr(g.depth_m, g.rmse_val)              # quantify the (likely weak) trend
ax.set_title(f'RMSE vs depth   (Spearman ρ = {rho:.2f}, p = {p:.2f})')
plt.tight_layout()
plt.savefig(figure_path + 'RMSEvsDepth.pdf')
plt.show()

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
for ax, ycol, ylab in [(a1, 'rmse_val', 'Validation RMSE [°C]'), (a2, 'nse_val', 'NSE')]:
    sc = ax.scatter(g.obs_std, g[ycol], c=g.abslat, cmap='inferno', s=45, edgecolor='white', lw=.4)
    rho, _ = spearmanr(g.obs_std, g[ycol])                 # now on the plotted variables
    ax.set_xlabel('Observed σ (°C)'); ax.set_ylabel(ylab)
    ax.set_title(f'{ylab.split(" [")[0]}   (Spearman ρ = {rho:.2f})')

a2.axhline(0.5, ls=':', c='gray')
a2.text(a2.get_xlim()[1], 0.5, ' satisfactory (0.5)', va='bottom', ha='right', fontsize=8, color='gray')

fig.colorbar(sc, ax=[a1, a2], label='|latitude| [°]', shrink=.8)
fig.suptitle('Model skill vs lake seasonal variability (σ of observed LSWT)', y=1.02)
plt.savefig(figure_path + 'skill_vs_sigma.pdf')
plt.show()


In [ ]:
res = spearmanr(g.obs_std, g.nse_val)
print('rho =', res.correlation, ' p =', res.pvalue)
res = spearmanr(g.obs_std, g.rmse_val)
print('rho =', res.correlation, ' p =', res.pvalue)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(metrics.rmse_cal, metrics.rmse_val, c=metrics.obs_std, cmap='inferno',
                s=45, edgecolor='white', lw=.4, zorder=5)

lims = [0, max(metrics.rmse_cal.max(), metrics.rmse_val.max()) * 1.05]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect('equal')

ax.set_xlabel('Calibration RMSE [°C]'); ax.set_ylabel('Validation RMSE [°C]')
plt.colorbar(sc, label='observed σ [°C]')
ax.set_title('Calibration vs validation RMSE (per lake)')
plt.savefig(figure_path + 'cal_vs_val_rmse.pdf')
plt.show()
print((metrics.rmse_val - metrics.rmse_cal).median())

In [ ]:
did  = 20853
base = a2w.DAMS_DIR / str(did) / str(did)
arr  = np.fromfile(glob.glob(str(base / 'output_3' / '0_PSO_*.out'))[0],
                   dtype=np.float64).reshape(-1, 9)
params = arr[:, :8]
rmse   = -arr[:, 8]                          # col 8 is -RMSE -> flip to RMSE
bounds = np.loadtxt(base / 'parameters.txt') # row0 = min, row1 = max
best   = arr[arr[:, 8].argmax(), :8]         # highest efficiency = lowest RMSE

names = [f'a{i+1}' for i in range(8)]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.plot(params[:, i], rmse, '.', ms=1.5, c='k', alpha=.15)   # the dotty cloud
    ax.axvline(bounds[0, i], ls='--', c='gray', lw=.8)           # lower bound
    ax.axvline(bounds[1, i], ls='--', c='gray', lw=.8)           # upper bound
    ax.axvline(best[i], c='tab:red', lw=1.3)                     # best value
    ax.set_xlabel(names[i]); ax.set_ylabel('RMSE [°C]')
fig.suptitle(f'Dotty plots — dam {did} (red = best set, dashed = a-priori bounds)')
plt.tight_layout()
plt.savefig(figure_path + f'dotty_{did}.pdf')
plt.show()

In [ ]:
# --- Headline results summary (Results, beats 1-3) -> CSV + LaTeX ---
gap = metrics.rmse_val - metrics.rmse_cal

def _pooled_rmse(period):
    d = long[long.period == period].dropna(subset=['obs_water', 'sim_water'])
    return np.sqrt(np.mean((d.sim_water - d.obs_water) ** 2)), len(d)

pooled_cal, n_cal = _pooled_rmse('cal')
pooled_val, n_val = _pooled_rmse('val')

deg = r'$^{\circ}$C'
summary = pd.DataFrame([
    ('Lakes calibrated and validated',            f'{len(metrics)} / {len(metrics)} (0 failures)'),
    ('Median calibration RMSE',                   f'{metrics.rmse_cal.median():.2f} {deg}'),
    ('Median validation RMSE',                    f'{metrics.rmse_val.median():.2f} {deg}'),
    ('Validation RMSE range',                     f'{metrics.rmse_val.min():.2f}--{metrics.rmse_val.max():.2f} {deg}'),
    ('Pooled daily RMSE, calibration',            f'{pooled_cal:.2f} {deg} ($n={n_cal:,}$)'),
    ('Pooled daily RMSE, validation',             f'{pooled_val:.2f} {deg} ($n={n_val:,}$)'),
    ('Median validation NSE',                     f'{metrics.nse_val.median():.2f}'),
    ('Median validation bias',                    f'{metrics.bias_val.median():+.2f} {deg}'),
    ('Median cal-to-val RMSE increase',           f'{gap.median():.2f} {deg} ({100*gap.median()/metrics.rmse_val.median():.0f}\\%)'),
    ('Lakes validating better than calibrating',  f'{int((gap < 0).sum())} / {len(metrics)}'),
], columns=['Metric', 'Value'])

# machine-readable CSV (LaTeX stripped)
plain = summary.copy()
plain['Value'] = (plain['Value'].str.replace(r'\$\^\{\\circ\}\$C', 'degC', regex=True)
                                .str.replace(r'[\\${}^]', '', regex=True))
plain.to_csv(a2w.HERE / 'results_summary_temp.csv', index=False)

# booktabs LaTeX table
body = ' \\\\\n'.join(f'{r.Metric} & {r.Value}' for r in summary.itertuples())
tex = ('\\begin{table}[t]\n\\centering\n'
       '\\caption{Headline validation results for air2water across the 56-lake set.}\n'
       '\\label{tab:temp_results_summary}\n'
       '\\begin{tabular}{ll}\n\\hline\n'
       'Metric & Value \\\\\n\\hline\n'
       f'{body} \\\\\n'
       '\\hline\n\\end{tabular}\n\\end{table}\n')
(a2w.HERE / 'results_summary_temp.tex').write_text(tex, encoding='utf-8')

summary

In [ ]:
#Naive air-temperature proxy vs air2water (validation period)
naive_rmse, a2w_rmse = [], []
for did, r in results.items():
    v = r['val_series'].dropna(subset=['obs_air', 'obs_water', 'sim_water'])
    o, a, s = v['obs_water'].to_numpy(), v['obs_air'].to_numpy(), v['sim_water'].to_numpy()
    naive_rmse.append(np.sqrt(np.mean((o - a) ** 2)))   # Tw = Tair, no model
    a2w_rmse.append(np.sqrt(np.mean((o - s) ** 2)))      # air2water
naive_rmse, a2w_rmse = np.array(naive_rmse), np.array(a2w_rmse)

print(f"Naive proxy (Tw=Tair) median RMSE: {np.median(naive_rmse):.2f} °C "
      f"(range {naive_rmse.min():.2f}-{naive_rmse.max():.2f})")
print(f"air2water median RMSE:             {np.median(a2w_rmse):.2f} °C")
print(f"Median improvement factor:         {np.median(naive_rmse / a2w_rmse):.1f}x")
print(f"air2water beats proxy in {int((a2w_rmse < naive_rmse).sum())}/{len(naive_rmse)} lakes")